In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
# integrate all process steps into a function
def raw2df(path1, path2, material, stroke, osc_freq, force):
    '''
    path1: the path of friction raw data
    path2: the path of AE raw data
    material: the material of the friction surface
    stroke: the stroke distance of the experiment
    osc_freq: the oscillation frequency of the experiment
    force: the normal force of the experiment   
    '''
    # preprocess friction raw data
    df1 = pd.read_csv(path1, sep='\\s+', header = None)
    df1 = df1[[0,2]]
    df1['coe'] = df1[2].apply(lambda x: x/10)
    df1.drop(columns=[2], inplace=True)
    df1 = df1.rename(columns={0: "Time"})

    # preprocess AE raw data
    df2 = pd.read_csv(path2, sep=',')
    df2.columns = df2.columns.str.strip()
    df2 = df2.rename(columns={"Time (*)": "Time"})
    df2 = df2[['Time', 'Absolute Energy', 'Amplitude', 'ASL', 'Average Frequency', 'Initiation Frequency', 'Parametric 1', 'Parametric 2', 'Peak Frequency', 'Reverberation Frequency', 'Threshold']]
    
    # replace timestamp with the number of cycles
    def timestamp_to_cycles(df3, col_name, frequency, df4 = pd.DataFrame()):
        '''
        df3: the dateframe that we want to preprocess
        col_name: the name of timestamp column
        frequency: the frequency of the experiment
        df4: a void dateframe that we will use to storage the processed dataframe
    
        '''
        df3['cycle'] = np.floor(df3[col_name] * frequency)+1
        df4 = df3.groupby('cycle').apply(lambda x: x.abs().mean(), include_groups=False).reset_index()  # use the mean absolute value to represent the feature in this cycle
        df4 = df4.drop(columns = [col_name])
        return df4
    df_friction = timestamp_to_cycles(df1, 'Time', osc_freq)
    df_ae = timestamp_to_cycles(df2, 'Time', osc_freq)

    # merge df_friction and df_ae
    df_prepared = pd.merge(df_friction, df_ae, on = 'cycle', how = 'outer')
    df_prepared = df_prepared.dropna(subset = ['coe'])
    df_prepared.fillna(0, inplace = True)

    # add experimental information
    df_prepared.insert(0, 'material', material) # add a column of experiment material
    df_prepared.insert(1, 'stroke', stroke) # add a column of stroke distance
    df_prepared.insert(2, 'osc_freq', osc_freq) # add a column of the oscillation frequency
    df_prepared.insert(3, 'force', force) # add a column of the normal force
    df_prepared['stage'] = 0 # add a column of the stage(label)
    return df_prepared


In [14]:
df = raw2df(r'D:\STUDY\internship\dataset\100_10_10_f.ASC', r'D:\STUDY\internship\dataset\100_10_10_AE.txt', 'steel-glass', 100, 10, 10)
df.head()

,material,stroke,osc_freq,force,cycle,coe,Absolute Energy,Amplitude,ASL,Average Frequency,Initiation Frequency,Parametric 1,Parametric 2,Peak Frequency,Reverberation Frequency,Threshold,stage
0,steel-glass,100,10,10,1.0,0.023768,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,steel-glass,100,10,10,2.0,0.024692,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,steel-glass,100,10,10,3.0,0.024137,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,steel-glass,100,10,10,4.0,0.024139,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,steel-glass,100,10,10,5.0,0.023798,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
# label the data according to the video